set up and data loading

In [ ]:
!pip install scanpy squidpy anndata igraph

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")

In [ ]:
adata.obs_names_make_unique()

In [ ]:
adata.var_names_make_unique()

### The AnnData object

The dataset has **4,035 spots × 36,601 genes** (obs = spots, var = genes). A Visium spot is 55 µm wide and mixes roughly 1–10 cells, so "spot" is used throughout instead of "cell". Scanpy function names like `filter_cells` are generic.

| Slot | Holds | In this dataset |
|---|---|---|
| `.X` | Main expression matrix | Raw UMI counts at load; normalised and log-transformed later |
| `.obs` | Per-spot metadata | `in_tissue`, `array_row`, `array_col` (grid position) |
| `.var` | Per-gene metadata | `gene_ids` (Ensembl), `feature_types`, `genome` (GRCh38) |
| `.obsm` | Per-spot matrices | `spatial` (pixel coordinates on the H&E image) |
| `.uns` | Unstructured metadata | `spatial`: hi/low-res images and scale factors for overlays |
| `.layers` | Alternative versions of `.X` | Empty at load; `counts` is added before normalisation |

Later steps add `.obs` columns (QC metrics, `leiden`), `.var` columns (`mt`, `highly_variable`), `.obsm["X_pca"]` and `["X_umap"]`, `.varm["PCs"]`, and neighbor graphs in `.obsp`.

In [ ]:
adata

In [ ]:
adata.obs["in_tissue"]

In [ ]:
adata.obs["array_row"]

In [ ]:
adata.obs["array_col"]

In [ ]:
adata.var["gene_ids"]

In [ ]:
adata.var["feature_types"]

In [ ]:
adata.var["genome"]

In [ ]:
adata.uns["spatial"]

In [ ]:
adata.obsm["spatial"]

In [ ]:
sq.pl.spatial_scatter(adata, color=None, size=0)

### Quality control

Per-spot metrics from `sc.pp.calculate_qc_metrics`: `total_counts` (UMIs), `n_genes_by_counts` (genes detected) and the percentage of counts from mitochondrial (`MT-`), ribosomal (`RPS/RPL`) and haemoglobin (`HB`) genes. High MT with low counts is the usual sign of damaged tissue, but some cell types are naturally MT-rich, so check where flagged spots sit before filtering.

**Violin plots** show each metric's distribution; use them to pick lower thresholds.
**Scatter (counts vs genes, coloured by MT)** shows how the metrics relate. A healthy sample gives a tight, saturating curve.
**Spatial plots** show whether low-count or high-MT spots form a region (local damage, fold, edge) or are scattered.

**What I found**
- Tight curve with no outlier population; ~8,000 genes at 30–40k counts.
- MT stays below ~5%, well under typical 10–20% cutoffs. Higher-MT spots sit on the lower edge of the curve at 5–20k counts.
- On the tissue: a compact MT-high group on the left and a low-count patch at the lower left.

**Decision: filter lightly.** `min_genes=100` removes 2 spots and `min_cells=3` removes 14,190 rarely detected genes (4,035 → 4,033 spots, 36,601 → 22,411 genes). Counts above 100 genes rise gradually (14 spots below 500, 27 below 1,000), so there is no sharp low-quality population and no stricter threshold was applied. No doublet detection: Visium spots are multi-cell by design.

**Limits:** the scatter cannot show tissue folds, edge artifacts or ambient RNA. The low-count and MT-high regions are revisited after clustering.

In [ ]:
# mitochondrial genes : "MT" for humans

adata.var["mt"] = adata.var_names.str.startswith("MT-")

adata.var["mt"]

In [ ]:
adata.var["mt"].sum()

In [ ]:
# ribosomal genes

adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

adata.var["ribo"]

In [ ]:
adata.var["ribo"].sum()

In [ ]:
# hemoglobin genes

adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

adata.var["hb"]

In [ ]:
adata.var["hb"].sum()

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars = ["mt", "ribo", "hb"], inplace = True, log1p = True)

In [ ]:
sc.pl.violin(adata,
             ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             jitter = 0.4,
             multi_panel = True
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color = "pct_counts_mt")

Observed: Higher-MT spots are spatially localised, not scattered: a coherent block on the central left of the tissue, plus smaller groups near the edges. Edge enrichment is consistent with handling or lysis damage during sectioning, but this is an interpretation, not something the QC plots prove. The central-left block suggests a localised tissue or capture effect (reduced depth with raised MT) or a region of MT-rich cell types, rather than diffuse degradation, because the rest of the tissue stays near 1% MT. MT stays low in absolute terms (max ~5%), so it does not indicate widespread damage. Clustering later isolated these spots (cluster 5: ~15,000 counts, 3.1% MT vs ~1% elsewhere), and the cause remains unresolved.

In [ ]:
sc.pl.spatial(adata, color=["total_counts", "pct_counts_mt"])

In [ ]:
for t in [10, 50, 100, 200, 500, 1000]:
    print(t, (adata.obs["n_genes_by_counts"] < t).sum())

In [ ]:
sc.pp.filter_cells(adata, min_genes = 100)
sc.pp.filter_genes(adata, min_cells = 3)

In [ ]:
# checkiing the obs/var counts
adata.n_obs, adata.n_vars

In [ ]:
# save a copy in the layer "counts"
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

### Highly variable genes and PCA

`highly_variable_genes(n_top_genes=2000)` keeps the genes that vary most across spots. PCA compresses them into components, each a weighted sum of genes. The variance-ratio plot shows how many components matter (here an elbow at ~5–10 PCs; `n_pcs=10` used for neighbors).

**Checks that PCs are biological, not technical**
- Colour PC scatters by `total_counts` and `pct_counts_mt`. A smooth gradient along a PC would mean depth or MT drives it.
- Map PC scores onto the tissue. Anatomical structure means real biology.
- Look at the gene loadings to see what each PC separates.

**What I found**
- No depth or MT gradient in PC1–PC4.
- PC1 maps to round, follicle-like patches. Positive loadings are germinal-centre/B-cell genes (*RGS13*, *MEF2B*, *CR2*), negative are stromal/endothelial/myeloid (*CCL14*, *ACKR1*, *C1QA*).
- PC2 separates T-cell genes (*CD247*, *ZAP70*, *TCF7*) from B-cell/stromal genes (*CXCL13*, *FN1*, *MARCO*).
- Plotting top-loading genes on the tissue reproduces the PC maps.

**Limits:** Visium spots mix cell types, so a PC axis blends them (the "follicle" axis includes B cells and follicular dendritic cells). Gene-to-cell-type readings come from literature and databases, not from this dataset.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes = 2000)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
# calculate total counts specifically for highly variable genes
hvg_genes = adata.var_names[adata.var["highly_variable"]]
adata.obs["hvg_total_counts"] = adata[:, hvg_genes].X.sum(axis=1)

# plotting spatially
sc.pl.spatial(adata, color=["hvg_total_counts"], ncols=1)

In [ ]:
sc.tl.pca(adata)

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs = 10, log = True)

In [ ]:
sc.pl.pca(adata,
          color = ["total_counts", "total_counts", "pct_counts_mt", "pct_counts_mt"],
          dimensions = [(0, 1), (2, 3), (0, 1), (2, 3)],
          ncols = 2,
          size = 2
          )

In [ ]:
adata.obs["PC1"] = adata.obsm["X_pca"][:, 0]
adata.obs["PC2"] = adata.obsm["X_pca"][:, 1]
sc.pl.spatial(adata, color=["PC1", "PC2"])

In [ ]:
sc.pl.pca_loadings(adata, components="1,2")

In [ ]:
adata.varm["PCs"]

In [ ]:
pca_loadings = pd.DataFrame(
    adata.varm["PCs"],
    index=adata.var_names,
    columns=[f"PC{i+1}" for i in range(adata.varm["PCs"].shape[1])],
)

# sort by component 1 (absolute value to catch both strongly positive and negative drivers)
top_pc1 = pca_loadings["PC1"].abs().sort_values(ascending=False).head(10)
print("--- Top Genes for PC1 ---")
print(top_pc1)

# Ssrt by component 2
top_pc2 = pca_loadings["PC2"].abs().sort_values(ascending=False).head(10)
print("\n--- Top Genes for PC2 ---")
print(top_pc2)

In [ ]:
top_pc1_list = (
    pca_loadings["PC1"].abs().sort_values(ascending=False).head(4).index.tolist()
)

sc.pl.spatial(adata, color=top_pc1_list, ncols=2)

### Neighbors, UMAP and clustering

`sc.pp.neighbors(n_pcs=10)` builds a k-nearest-neighbor graph in PC space, `sc.tl.umap` embeds it in 2D for display, and Leiden (igraph, `n_iterations=3`, resolution 1.0, `random_state=0`) partitions the graph into clusters. UMAP shows transcriptional similarity only, so spots can sit together on the UMAP while far apart on the tissue. Clustering uses expression, not coordinates.

**Result:** 15 clusters. QC metrics show no depth or MT structure on the UMAP except for the technical clusters noted below.

**Robustness:** an earlier run gave 13 clusters and this run gave 15; the version/environment differed. Core compartments (plasma cells, T zone, sinus, follicle/mantle, germinal centre, stroma, interferon-response region) appear in both, while finer splits (e.g. two plasma-cell and two T-cell clusters) do not. Cluster IDs are arbitrary, so labels apply only to this exact run.

In [ ]:
sc.pp.neighbors(adata, n_pcs = 10, random_state = 42)

In [ ]:
sc.tl.umap(adata, random_state = 42)

In [ ]:
sc.tl.leiden(adata, flavor = "igraph", n_iterations = 3, random_state = 42)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.pl.spatial(adata, color=["leiden"])

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "n_genes_by_counts", "pct_counts_mt"], ncols=2)

In [ ]:
sc.pl.umap(adata, color=["RGS13", "MEF2B", "CD247", "CCL21", "ACKR1", "C1QA"], vmax="p99", ncols=3)

In [ ]:
sc.pl.spatial(adata, color=["RGS13", "MEF2B", "CD247", "CCL21", "ACKR1", "C1QA"], vmax="p99", ncols=3)

In [ ]:
adata.obs["x"] = adata.obsm["spatial"][:, 0]
adata.obs["y"] = adata.obsm["spatial"][:, 1]
sc.pl.umap(adata, color=["x", "y"])

In [ ]:
sc.tl.score_genes(adata, ["MS4A1", "CD79A", "PAX5", "CD19"], score_name="B_score")
sc.tl.score_genes(adata, ["CD3E", "CD3D", "ZAP70", "TCF7"], score_name="T_score")
sc.pl.umap(adata, color=["B_score", "T_score"])

In [ ]:
sc.pl.spatial(adata,
              color=["leiden", "total_counts", "n_genes_by_counts", "pct_counts_mt" ],
              ncols=3,
              vmax="p99",)

In [ ]:
sc.pl.spatial(adata,
              color=["B_score", "T_score"],
              ncols=3,
              vmax="p99",)

### Marker genes and annotation

`rank_genes_groups` (Wilcoxon, one cluster vs the rest) lists the genes enriched in each cluster. Labels come from matching the top markers to known lymph node compartments from the literature, and from checking each cluster's sequencing depth and mitochondrial fraction. Labels are **provisional**: they rest on marker genes and published gene functions, and have not been reviewed by a pathologist or immunologist.

| Cluster | Label | Key markers | Confidence |
|---|---|---|---|
| 0, 14 | Plasma cells | *IGHG1*, *IGKC*, *XBP1*, *JCHAIN*, *MZB1* | high |
| 2, 12 | T zone (T cells; 12 more stroma-rich) | *TRAC*, *TRBC1/2*, *CCL19*, *CD3E*, *CCR7* | high |
| 3 | Sinus / lymphatic endothelium | *CLEC4M*, *CD209*, *LYVE1*, *STAB1*, *MARCO* | high |
| 4 | Fibroblast-like stroma | *CCL20*, *SERPINE1*, *CHI3L1*, *IGFBP7* | medium |
| 6 | T-zone stroma / vascular (mixed) | *CCL21*, *SPARCL1*, *CCL2*, *VWF* | medium |
| 7 | Interferon response | *ISG15*, *MX1*, *IFI6*, *OAS2/3* | high |
| 8 | Follicle / mantle + FDC | *CXCL13*, *MS4A1*, *CD19*, *CR2* | medium |
| 10 | Follicle, GC-adjacent (mixed) | *CR2*, *TCL1A*, *FDCSP*; shares its markers with 11 | low |
| 11 | Germinal centre | *MEF2B*, *RGS13*, *HMGB2*, *LRMP* | high |
| 9 | Vascular / connective tissue (?) | *MGP*, *MYL9*, *IGFBP4*; immunoglobulin genes also present | low |
| 1 | Unresolved (mixed myeloid/stromal) | *ITGAX*, *DNASE1L3*, *LUM*, *TNC* | low |
| 5 | MT-high, lower depth (technical?) | MT genes; 15,019 counts, 3.1% MT | medium |
| 13 | Low-count (technical) | housekeeping/ribosomal genes; 1,184 counts | high |

**Notes**
- Marker-based scores partly reuse the genes that defined the clusters, so agreement is a consistency check, not independent evidence.
- Immunoglobulin transcripts are abundant and spread between spots, so plasma-cell signal can appear in neighbouring clusters (for example cluster 9).
- Clusters 5 and 13 are flagged technical from the QC table (depth and MT fraction). They are kept in the data and excluded from biological interpretation.
- Spatial maps and the H&E image were used only as a sanity check that clusters are spatially coherent. Anatomical interpretation of the tissue was not attempted without expert review.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")

In [ ]:
top_genes = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).iloc[0]
print(top_genes)  # index = cluster, value = top gene

top_gene_list = top_genes.tolist()

In [ ]:
sc.pl.umap(adata, color=top_gene_list, ncols=4, vmax="p99")

In [ ]:
sc.pl.spatial(adata, color=top_gene_list, ncols=4, vmax="p99")

In [ ]:
# top 5 genes per cluster as a dotplot
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, standard_scale="var")

# heatmap alternative
sc.pl.rank_genes_groups_heatmap(adata, n_genes=5, show_gene_labels=True)

# ranked-gene panels with scores
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

In [ ]:
sc.get.rank_genes_groups_df(adata, group="0").head(10)  # one cluster

In [ ]:
markers = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
print(markers)

In [ ]:
# several genes appear in more than one cluster's top 10. they make poor signature genes since they cannot separate the clusters. hence filter for specific markers first
sc.tl.filter_rank_genes_groups(
    adata, min_in_group_fraction=0.25, max_out_group_fraction=0.5, min_fold_change=1.5
)

sc.pl.rank_genes_groups_dotplot(
    adata, n_genes=4, key="rank_genes_groups_filtered", standard_scale="var"
)

In [ ]:
adata.obs.groupby("leiden")[["total_counts","n_genes_by_counts","pct_counts_mt"]].mean().round(1)
sc.pl.spatial(adata, color="leiden", groups=["5","13"], na_color="lightgrey", size=1.3)
for a, b in [("0","14"), ("2","12"), ("10","11")]:
    print(a, "vs", b)
    sc.tl.rank_genes_groups(adata, "leiden", groups=[a], reference=b, method="wilcoxon", key_added=f"rgg_{a}_{b}")
    print(sc.get.rank_genes_groups_df(adata, group=a, key=f"rgg_{a}_{b}").head(8)[["names","scores","logfoldchanges"]])

In [ ]:
signatures = {
    "Plasma cells":        ["IGHG1", "IGKC", "JCHAIN", "XBP1", "SSR4", "MZB1"],
    "T zone (T cells)":    ["TRAC", "TRBC1", "CD3E", "CD3D", "IL7R", "TCF7", "CCL19"],
    "T-zone stroma":       ["CCL21", "CCL2", "CCN1", "SPARCL1", "CEBPD"],
    "Sinus/lymphatic":     ["CLEC4M", "CD209", "LYVE1", "STAB1", "CLEC4G", "MARCO"],
    "Interferon response": ["ISG15", "MX1", "IFI6", "IFI44", "OAS2", "STAT1"],
    "Follicle/mantle":     ["MS4A1", "CD22", "FCER2", "CD19", "TNFRSF13C"],
    "FDC network":         ["CXCL13", "FDCSP", "CR2"],
    "Germinal center":     ["RGS13", "MEF2B", "LRMP", "ELL3", "SERPINE2", "HMGB2"],
    "Fibroblast-like":     ["VIM", "SERPINE1", "CHI3L1", "IGFBP7", "TIMP1", "CCL20"],
}

for name, genes in signatures.items():
    genes = [g for g in genes if g in adata.var_names]
    sc.tl.score_genes(adata, genes, score_name=name)

score_table = adata.obs.groupby("leiden")[list(signatures)].mean().round(2)
print(score_table)

In [ ]:
sc.pl.dotplot(adata, signatures, groupby="leiden", standard_scale="var")

In [ ]:
adata.obs.groupby("leiden")[["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo"]].mean().round(1)

In [ ]:
sc.pl.spatial(adata, color = "leiden", groups = ["9"], na_color="black")

**Label versions (provisional → final)**
- `compartment_run2`: first-pass labels, made from the top markers only (before checking depth and MT per cluster).
- `compartment_run3`: **final labels**, updated after the QC table showed that clusters 5 and 13 are technical and that cluster 9 has normal depth. Downstream cells use this column and `is_technical`.
The first-pass plot is kept to show what changed; do not use it for interpretation.

In [ ]:
cluster_labels_run2 = {
    "0": "Plasma cells",
    "1": "Unresolved (mixed myeloid/stromal)",
    "2": "T zone (T cells)",
    "3": "Sinus / lymphatic endothelium",
    "4": "Fibroblast-like stroma",
    "5": "MT-high (technical?)",
    "6": "T-zone stroma / vascular (mixed)",
    "7": "Interferon response",
    "8": "Follicle / mantle + FDC",
    "9": "Vessel-associated (?) + Ig spillover",
    "10": "Follicle, GC-like (mixed)",
    "11": "Germinal center",
    "12": "T cells + T-zone stroma (mixed)",
    "13": "Low-count region (technical)",
    "14": "Plasma cells (2)",
}
adata.obs["compartment_run2"] = adata.obs["leiden"].map(cluster_labels_run2).astype("category")

In [ ]:
print(adata.obs["compartment_run2"].isna().sum())   # should be 0
print(adata.obs["compartment_run2"].value_counts())

In [ ]:
sc.pl.spatial(adata, color="compartment_run2")

In [ ]:
sc.pl.spatial(adata, color="leiden", groups=["4"], na_color="lightgrey", size=1.3)
sc.pl.spatial(adata, color="leiden", groups=["6", "12"], na_color="lightgrey", size=1.3)
sc.pl.spatial(adata, color="leiden", groups=["0", "3"], na_color="lightgrey", size=1.3)

In [ ]:
cluster_labels_run2.update({
    "0": "Plasma cells (mixed)",
    "14": "Plasma cells (dense)",
    "5": "MT-high, lower depth (technical?)",
    "9": "Vascular / connective tissue (hilum, capsule?)",
    "13": "Low-count region (technical)",
})
adata.obs["is_technical"] = adata.obs["leiden"].isin(["5", "13"])

In [ ]:
adata.obs["compartment_run3"] = adata.obs["leiden"].map(cluster_labels_run2).astype("category")

sc.pl.spatial(adata, color="compartment_run3")

sc.pl.spatial(adata, color="is_technical")

In [ ]:
adata.obs["is_technical"].sum()

### Pathway enrichment and gene-set scoring

Two complementary analyses:
- **Preranked GSEA** (`gseapy`, MSigDB Hallmark): ranks all genes by their score for a cluster (one cluster vs the rest) and asks which pathways sit towards the top or bottom of the ranking, without choosing one in advance.
- **Gene-set scoring** (`sc.tl.score_genes`): gives each spot a score for a chosen set (e.g. Hallmark G2-M, interferon alpha/gamma), which can be mapped on the tissue. No statistical test; compare groups.

**Results**
- **Germinal centre (cluster 11):** proliferation programmes are enriched: E2F targets (NES 2.19), G2-M checkpoint (NES 2.09) and Myc targets V1 (NES 2.02), with lead genes such as *TOP2A*, *PCNA*, *PTTG1* and *STMN1*. The G2/M score is highest in this cluster (mean 0.40) and separates it from all other spots with an AUC of 0.97, and from the GC-adjacent cluster 10 with an AUC of 0.86. Consistent with rapid B-cell division in germinal centres.

- **Interferon region (cluster 7):** interferon alpha response is the top GSEA term (NES 2.26) and interferon gamma response ranks third (NES 2.10), led by *ISG15*, *MX1*, *IFI44L* and *OAS* genes. Myc targets V1 (2.07), E2F targets (1.84) and G2-M (1.72) are also positive. The curated Hallmark scores agree: mean IFNa is 0.85 in this cluster against 0.34–0.47 in all other clusters (0.21 in the low-count cluster 13), and mean IFNg is 0.75 against 0.40–0.54. The cause cannot be determined from one section.

**Limits:** p-values are inflated by spatial correlation, so rank by NES and report FDR as < 0.001 instead of 0. About 30% of genes had tied scores in the cluster 7 run, so their order is arbitrary. Negative enrichment (e.g. interferon and inflammatory terms in cluster 11) mostly reflects the cell composition of the rest of the tissue, not suppression. The G2/M score is supporting evidence, not independent validation, since cell-cycle genes overlap with germinal centre markers.

In [ ]:
!pip install gseapy

In [ ]:
sc.pl.spatial(adata, color="leiden", groups=["7"], na_color="black", size=1.3)

In [ ]:
ifn_genes = ["ISG15", "MX1", "IFI6", "IFI44", "IFI44L", "OAS2", "OAS3", "STAT1", "XAF1", "MX2"]
ifn_genes = [g for g in ifn_genes if g in adata.var_names]

sc.tl.score_genes(adata, ifn_genes, score_name="IFN_score")
sc.pl.spatial(adata, color="IFN_score", vmax="p99")

In [ ]:
import gseapy as gp
hallmark = gp.get_library("MSigDB_Hallmark_2020")  # needs internet

for term, name in [("Interferon Alpha Response", "IFNa"),
                   ("Interferon Gamma Response", "IFNg")]:
    genes = [g for g in hallmark[term] if g in adata.var_names]
    sc.tl.score_genes(adata, genes, score_name=name)

sc.pl.spatial(adata, color=["IFNa", "IFNg"], vmax="p99")

In [ ]:
adata.obs.groupby("leiden")[["IFN_score", "IFNa", "IFNg"]].mean().round(2)

In [ ]:
print(gp.get_library_name())

In [ ]:
hallmark = gp.get_library("MSigDB_Hallmark_2020")  # needs internet
g2m_genes = [g for g in hallmark["G2-M Checkpoint"] if g in adata.var_names]
sc.tl.score_genes(adata, g2m_genes, score_name="G2M")

In [ ]:
# peranked GSEA: rerun for the new GC cluster and for the interferon one
for grp in ["11", "7"]:
    df = sc.get.rank_genes_groups_df(adata, group=grp)
    ranked = df.set_index("names")["scores"].sort_values(ascending=False)
    pre = gp.prerank(rnk=ranked, gene_sets="MSigDB_Hallmark_2020", outdir=None, seed=0)
    print(grp)
    print(pre.res2d.head(10))

In [ ]:
sc.pl.spatial(adata, color="leiden", groups=["11", "7"], size=1.3, vmax="p99")

In [ ]:
adata.obs["is_GC"] = adata.obs["leiden"].isin(["11"])
print(adata.obs.groupby("leiden")["G2M"].mean().sort_values(ascending=False))

In [ ]:
sc.pl.spatial(adata, color="leiden", groups=["10", "11"], size=1.3, vmax="p99")

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
adata.obs["is_GC"] = adata.obs["leiden"] == "11"
print(roc_auc_score(adata.obs["is_GC"], adata.obs["G2M"]))

sub = adata.obs[adata.obs["leiden"].isin(["11", "10"])]      # GC vs GC-adjacent
print(roc_auc_score(sub["leiden"] == "11", sub["G2M"]))

In [ ]:
sc.pl.violin(adata, "G2M", groupby="leiden")

In [ ]:
sc.pl.dotplot(adata, ["MKI67", "TOP2A", "RGS13", "BCL6", "CXCR4", "CD83"],
              groupby="leiden", standard_scale="var")